In [25]:
import sqlite3
import pandas as pd

# Create database
conn = sqlite3.connect("grocchat.db")

# Load CSV file
df = pd.read_csv("final_dataset.csv")

# Preview data
df.head()

,product_id,product_name,supplier_id,supplier_name,stock_left,last_time_ordered_date,received_date,sale_volume_previous_month,status,reorder_level,unit_price_usd,category,supplier_lead_time_days,expiry_date,sales_week1,sales_week2,sales_week3,sales_week4
0,PROD001,Lakshmi Sooji 4lb,S004,Oil India Traders,81,2/22/2026,3/3/2026,993,active,77,3.25,General,9,4/9/2026,179,196,262,356
1,PROD002,MDH Sooji 4lb,S002,Spice World India,121,2/20/2026,2/21/2026,1077,active,177,3.40,General,1,4/6/2026,324,301,179,273
2,PROD003,Patanjali Sooji 4lb,S004,Oil India Traders,239,2/26/2026,2/27/2026,1196,active,157,3.10,General,1,3/30/2026,272,314,395,215
3,PROD004,Bikaji Sooji 4lb,S005,Haldiram Distributors,89,2/25/2026,3/3/2026,557,active,161,3.55,General,6,4/20/2026,107,154,135,161
4,PROD005,Deep Besan 4lb,S008,MTR Ready Mix,284,2/13/2026,2/15/2026,915,active,162,3.35,General,2,3/6/2026,216,194,217,288


In [26]:
df.to_sql("inventory", conn, if_exists="replace", index=False)

print("Data stored in database!")

Data stored in database!


In [27]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")


In [28]:
pd.read_sql("SELECT * FROM inventory LIMIT 5", conn)

,product_id,product_name,supplier_id,supplier_name,stock_left,last_time_ordered_date,received_date,sale_volume_previous_month,status,reorder_level,unit_price_usd,category,supplier_lead_time_days,expiry_date,sales_week1,sales_week2,sales_week3,sales_week4
0,PROD001,Lakshmi Sooji 4lb,S004,Oil India Traders,81,2/22/2026,3/3/2026,993,active,77,3.25,General,9,4/9/2026,179,196,262,356
1,PROD002,MDH Sooji 4lb,S002,Spice World India,121,2/20/2026,2/21/2026,1077,active,177,3.40,General,1,4/6/2026,324,301,179,273
2,PROD003,Patanjali Sooji 4lb,S004,Oil India Traders,239,2/26/2026,2/27/2026,1196,active,157,3.10,General,1,3/30/2026,272,314,395,215
3,PROD004,Bikaji Sooji 4lb,S005,Haldiram Distributors,89,2/25/2026,3/3/2026,557,active,161,3.55,General,6,4/20/2026,107,154,135,161
4,PROD005,Deep Besan 4lb,S008,MTR Ready Mix,284,2/13/2026,2/15/2026,915,active,162,3.35,General,2,3/6/2026,216,194,217,288


In [29]:
pd.read_sql("""
SELECT product_name, stock_left, reorder_level
FROM inventory
WHERE stock_left < reorder_level
""", conn)

,product_name,stock_left,reorder_level
0,MDH Sooji 4lb,121,177
1,Bikaji Sooji 4lb,89,161
2,Pillsbury Besan 4lb,108,129
3,Lakshmi Atta 10lb,43,187
4,Konark Ghee 4lb,122,154
5,ITC Rajma 4lb,94,194
6,Tata Rajma 4lb,107,150
7,Konark Moong Dal 4lb,44,80
8,Lakshmi Masoor Dal 4lb,114,146
9,MTR Idli Mix 2lb,75,96


In [30]:
!pip install openai


In [31]:
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key="FNos7JA0VT6gvlwHsc0VE71d5KZCy6JzAou519avDPaVKU4qrfcgJQQJ99CDACfhMk5XJ3w3AAAAACOGFeQF",
    api_version="2024-12-01-preview",
    azure_endpoint="https://grocchaatt-resource.cognitiveservices.azure.com/"
)

In [32]:
response = client.chat.completions.create(
    model="gpt-4o-mini",  # your deployment name
    messages=[{"role": "user", "content": "Say hello"}]
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


In [33]:
schema = """
Table: inventory

Columns:
product_id
product_name
supplier_id
supplier_name
stock_left
reorder_level
sale_volume_previous_week
sale_volume_previous_month
expiry_date
category
supplier_lead_time_days
sales_week1
sales_week2
sales_week3
sales_week4
"""

In [34]:
def generate_sql(question):

    prompt = f"""
You are a SQL assistant.

Database schema:
{schema}

Rules:
- Use only the inventory table
- Generate only SELECT queries
- Do not modify the database
- Return only SQL (no explanation)

IMPORTANT:
- expiry_date is stored as DATE
- Use SQLite date functions
- To filter by month, use: strftime('%m', expiry_date)

User Question:
{question}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [35]:
def clean_sql(sql_query):
    return sql_query.replace("```sql", "").replace("```", "").strip()


def run_sql(sql_query):
    sql_query = clean_sql(sql_query)

    print("Generated SQL:")
    print(sql_query)

    if not sql_query.lower().startswith("select"):
        print("Only SELECT queries allowed")
        return None

    return pd.read_sql(sql_query, conn)

In [36]:
df['expiry_date'] = pd.to_datetime(df['expiry_date'])
df['expiry_date'] = df['expiry_date'].dt.strftime('%Y-%m-%d')

df.to_sql("inventory", conn, if_exists="replace", index=False)

100

In [37]:
!pip install faiss-cpu sentence-transformers
!pip install ipywidgets

In [38]:
promotion_chunks = [
    "Offer discounts for near expiry products to reduce loss",
    "Bundle slow-moving items with fast-selling products",
    "Apply small discounts for high demand products to maintain margin",
    "Use higher discounts for low demand products to clear inventory",
    "Promote sweets and snacks during festival seasons",
    "Use limited-time offers to create urgency",
    "Place high demand items at visible locations to increase sales"
]

In [39]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(promotion_chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [40]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

In [41]:
def retrieve_knowledge(query, k=3):

    query_embedding = model.encode([query])
    
    distances, indices = index.search(np.array(query_embedding), k)
    
    results = [promotion_chunks[i] for i in indices[0]]
    
    return "\n".join(results)

In [42]:
def decision_engine(df):

    decisions = []

    for _, row in df.iterrows():

        # 🔴 Low stock
        if 'stock_left' in df.columns and 'reorder_level' in df.columns:
            if row['stock_left'] < row['reorder_level']:
                decisions.append(f"Restock {row['product_name']} (low stock)")

        # 🟡 Expiry logic (only if column exists)
        if 'expiry_date' in df.columns:
            if pd.to_datetime(row['expiry_date']) <= pd.Timestamp.today() + pd.Timedelta(days=7):
                decisions.append(f"Apply discount on {row['product_name']} (expiring soon)")

        # 📈 Trend logic
        if 'sales_week1' in df.columns and 'sales_week4' in df.columns:
            trend = row['sales_week4'] - row['sales_week1']
            if trend > 10:
                decisions.append(f"Increase stock for {row['product_name']} (demand rising)")

    return decisions

In [43]:
def explain_decisions(df, decisions, question):

    relevant_knowledge = retrieve_knowledge(question)
    data_summary = df.to_string(index=False)

    prompt = f"""
You are an AI assistant for a grocery store owner.

DATA:
{data_summary}

DECISIONS:
{decisions}

RELEVANT KNOWLEDGE:
{relevant_knowledge}

Instructions:
- Be concise and business-focused
- No generic advice
- Use ONLY the provided data
- Group products logically (high demand, low demand, expiring)
- Suggest realistic actions (discount %, bundling, etc.)
- Maximum 5 bullet points total
- Write like a store advisor

Output format:

📊 Key Insights:
- ...

📌 Recommended Actions:
- ...
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [44]:
def ask_ai(question):

    # Step 1: Generate SQL
    sql_query = generate_sql(question)

    # Step 2: Run SQL
    result = run_sql(sql_query)

    if result is None or result.empty:
        print("No data found")
        return

    # Step 3: Show data
    print("\nData:")
    display(result)

    # Step 4: Create decisions (IMPORTANT)
    decisions = decision_engine(result)

    print("\nRaw Decisions:")
    for d in decisions:
        print(d)

    # Step 5: Explanation (FIXED)
    explanation = explain_decisions(result, decisions, question)

    print("\nAI Explanation:")
    print(explanation)

In [45]:
ask_ai("which products will expire in april?")

Generated SQL:
SELECT *
FROM inventory
WHERE strftime('%m', expiry_date) = '04';

Data:


,product_id,product_name,supplier_id,supplier_name,stock_left,last_time_ordered_date,received_date,sale_volume_previous_month,status,reorder_level,unit_price_usd,category,supplier_lead_time_days,expiry_date,sales_week1,sales_week2,sales_week3,sales_week4
0,PROD001,Lakshmi Sooji 4lb,S004,Oil India Traders,81,2/22/2026,3/3/2026,993,active,77,3.25,General,9,2026-04-09,179,196,262,356
1,PROD002,MDH Sooji 4lb,S002,Spice World India,121,2/20/2026,2/21/2026,1077,active,177,3.40,General,1,2026-04-06,324,301,179,273
2,PROD004,Bikaji Sooji 4lb,S005,Haldiram Distributors,89,2/25/2026,3/3/2026,557,active,161,3.55,General,6,2026-04-20,107,154,135,161
3,PROD006,Pillsbury Besan 4lb,S010,Deep Foods Ltd,108,2/2/2026,2/6/2026,1143,active,129,3.80,General,4,2026-04-02,238,229,249,427
4,PROD011,Lakshmi Urad Dal 4lb,S004,Oil India Traders,391,2/9/2026,2/19/2026,1335,active,126,3.10,Pulses,10,2026-04-13,466,210,215,444
5,PROD012,Everest Urad Dal 4lb,S003,Dal Masters Pvt Ltd,270,2/25/2026,2/26/2026,1293,active,110,3.35,Pulses,1,2026-04-07,312,339,382,260
6,PROD015,MTR Toor Dal 4lb,S001,Rajesh Grocery Suppliers,498,2/24/2026,3/1/2026,622,active,97,3.25,Pulses,5,2026-04-18,186,154,124,158
7,PROD018,Bikaji Mustard Oil 5liter,S002,Spice World India,384,2/3/2026,2/12/2026,1220,active,191,22.00,Cooking Oil,9,2026-04-08,421,284,361,154
8,PROD019,Lakshmi Ghee 4lb,S004,Oil India Traders,485,2/24/2026,2/28/2026,1349,active,165,48.00,General,4,2026-04-21,397,457,225,270
9,PROD021,Konark Ghee 4lb,S010,Deep Foods Ltd,122,2/23/2026,2/24/2026,700,active,154,47.80,General,1,2026-04-09,125,138,205,232



Raw Decisions:
Apply discount on Lakshmi Sooji 4lb (expiring soon)
Increase stock for Lakshmi Sooji 4lb (demand rising)
Restock MDH Sooji 4lb (low stock)
Apply discount on MDH Sooji 4lb (expiring soon)
Restock Bikaji Sooji 4lb (low stock)
Increase stock for Bikaji Sooji 4lb (demand rising)
Restock Pillsbury Besan 4lb (low stock)
Apply discount on Pillsbury Besan 4lb (expiring soon)
Increase stock for Pillsbury Besan 4lb (demand rising)
Apply discount on Everest Urad Dal 4lb (expiring soon)
Apply discount on Bikaji Mustard Oil 5liter (expiring soon)
Restock Konark Ghee 4lb (low stock)
Apply discount on Konark Ghee 4lb (expiring soon)
Increase stock for Konark Ghee 4lb (demand rising)
Restock ITC Rajma 4lb (low stock)
Apply discount on ITC Rajma 4lb (expiring soon)
Increase stock for ITC Rajma 4lb (demand rising)
Restock Lakshmi Masoor Dal 4lb (low stock)
Apply discount on Lakshmi Masoor Dal 4lb (expiring soon)
Increase stock for Lakshmi Masoor Dal 4lb (demand rising)
Apply discount on 